<a href="https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
print("Ready. Month =", MONTH)

Ready. Month = 2026-03


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Building a feature vector for the Refresh / Content Opportunity Scoring lane,
using only signals observed within the feature month itself — no future data,
no product flags.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

raw = con.sql(f"""
    SELECT content_hash_id,
           MAX(client_hash_id) AS client_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           AVG(gsc_avg_position) AS avg_position,
           SUM(ga4_sessions) AS sessions,
           SUM(ga4_engaged_sessions) AS engaged_sessions,
           BOOL_OR(ga4_data_available IS TRUE) AS has_ga4
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 50
""").df()

raw["ctr"] = raw["clicks"] / raw["impressions"]

def position_tier(pos):
    if pos <= 3: return "1-3"
    elif pos <= 10: return "4-10"
    elif pos <= 20: return "11-20"
    elif pos <= 50: return "21-50"
    else: return "51+"

raw["position_tier"] = raw["avg_position"].apply(position_tier)

def impression_tier(imp):
    if imp < 500: return "low"
    elif imp < 5000: return "mid"
    else: return "high"

raw["impression_tier"] = raw["impressions"].apply(impression_tier)

feature_vector = pd.get_dummies(raw, columns=["position_tier", "impression_tier"], prefix=["pos", "imp"])

feature_vector["sessions"] = feature_vector["sessions"].fillna(0)
feature_vector["engaged_sessions"] = feature_vector["engaged_sessions"].fillna(0)

print(f"Feature vector shape: {feature_vector.shape}")
feature_vector.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (116114, 17)


,content_hash_id,client_hash_id,impressions,clicks,avg_position,sessions,engaged_sessions,has_ga4,ctr,pos_1-3,pos_11-20,pos_21-50,pos_4-10,pos_51+,imp_high,imp_low,imp_mid
0,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,899.0,1.0,5.145765,0.0,0.0,False,0.001112,False,False,False,True,False,False,False,True
1,content_d49a012dcb924e31,client_62f4a7e64f5e0096,329.0,0.0,5.177774,0.0,0.0,False,0.000000,False,False,False,True,False,False,True,False
2,content_614baf2af4330bd7,client_62f4a7e64f5e0096,772.0,1.0,4.685335,0.0,0.0,False,0.001295,False,False,False,True,False,False,False,True
3,content_225dc9235023be5f,client_62f4a7e64f5e0096,488.0,1.0,17.148172,0.0,0.0,False,0.002049,False,True,False,False,False,False,True,False
4,content_babcf791dccc1610,client_62f4a7e64f5e0096,181.0,0.0,10.479603,0.0,0.0,False,0.000000,False,True,False,False,False,False,True,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before decision point? |
|---|---|---|---|
| `impressions` | GSC impressions summed over the feature month | none expected (filtered to >=50) | Yes — observed search data |
| `clicks` | GSC clicks summed over the feature month | none expected | Yes |
| `avg_position` | average search ranking position | none expected | Yes |
| `ctr` | clicks/impressions, derived | none (denominator filtered >0) | Yes — derived only from same-month data |
| `sessions` | GA4 sessions | filled 0 where GA4 unavailable (~96% of rows, per Week 3) | Yes where available; 0-fill is a real limitation, not true zero |
| `engaged_sessions` | GA4 engaged sessions | same as sessions | Same caveat |
| `has_ga4` | flag: was GA4 tracking on for this page/client | never missing (boolean) | Yes — a context flag, not a performance signal |
| `position_tier` (one-hot) | bucketed avg_position | derived, no missing | Yes |
| `impression_tier` (one-hot) | bucketed impressions | derived, no missing | Yes |

**Note on the GA4 zero-fill:** filling missing GA4 sessions with 0 is a known
limitation — it conflates "genuinely zero sessions" with "GA4 wasn't tracking
this page at all." This was the exact issue that hurt model performance in
Week 5, and it's flagged here explicitly rather than hidden.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

missing_report = feature_vector[["impressions", "clicks", "avg_position", "ctr", "sessions", "engaged_sessions"]].isna().sum()
print("Missing value counts after fills:")
print(missing_report)
print(f"\nRows where GA4 was unavailable (sessions/engaged_sessions filled with 0): {(~feature_vector['has_ga4']).sum()} of {len(feature_vector)}")

Missing value counts after fills:
impressions         0
clicks              0
avg_position        0
ctr                 0
sessions            0
engaged_sessions    0
dtype: int64

Rows where GA4 was unavailable (sessions/engaged_sessions filled with 0): 57435 of 116114


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attacking the feature vector directly: checking for label-derived columns,
future-window contamination, and product flags.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

forbidden = ["health_score", "priority_score", "action_type", "refresh_tier"]
present_forbidden = [c for c in forbidden if c in feature_vector.columns]
print(f"Test 1 — forbidden product flags present: {present_forbidden if present_forbidden else 'NONE — pass'}")

month_check = con.sql(f"""
    SELECT DISTINCT month FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
print(f"\nTest 2 — distinct months touched by this feature build: {month_check['month'].tolist()}")
print("Only one month present confirms no future-window data entered the feature vector.")

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

y_toy = (feature_vector["ctr"] < feature_vector["ctr"].median()).astype(int)
X_honest = feature_vector[["impressions", "avg_position", "sessions", "engaged_sessions"]].fillna(0)

honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y_toy)
honest_score = precision_score(y_toy, honest.predict(X_honest))
print(f"\nTest 3 — honest precision (ctr excluded from features, since label is built from ctr): {honest_score:.3f}")

X_leaky = X_honest.copy()
X_leaky["ctr"] = feature_vector["ctr"]
leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y_toy)
leaky_score = precision_score(y_toy, leaky.predict(X_leaky))
print(f"Leaky precision (ctr re-introduced, jumps toward perfect): {leaky_score:.3f}")
del X_leaky
print(f"Leak identified and removed. Keeping the honest number: {honest_score:.3f}")

Test 1 — forbidden product flags present: NONE — pass

Test 2 — distinct months touched by this feature build: ['2026-03']
Only one month present confirms no future-window data entered the feature vector.

Test 3 — honest precision (ctr excluded from features, since label is built from ctr): 0.769
Leaky precision (ctr re-introduced, jumps toward perfect): 1.000
Leak identified and removed. Keeping the honest number: 0.769


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| `health_score`, `priority_score`, `action_type`, `refresh_tier` | FlyRank's own product decisions, not observable signals — using them would let a model copy the existing rule instead of finding real signal (circular result). |
| `client_hash_id`, `content_hash_id` as model inputs | Join keys/identifiers, not performance signal — used only for grouping and joins. |
| Raw `report_date` as a model input | A row-level timestamp, not itself a meaningful feature at the page-month grain used here; date structure is handled through the month-window design instead. |
| GA4 sessions as the *sole* justification for a feature | Only ~4% GA4 coverage in this data (Week 3 finding) — included, but flagged with `has_ga4` since the 0-fill is a known weakness, not a true zero. |
| Any future-month data | Excluded entirely from this feature build — this notebook uses only `month={MONTH}`, no later window touched. |

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

excluded = ["health_score", "priority_score", "action_type", "refresh_tier"]
print(f"Confirmed excluded from feature_vector.columns: {[c for c in excluded if c not in feature_vector.columns]}")
print(f"\nFeature vector final columns: {[c for c in feature_vector.columns if c not in ['content_hash_id','client_hash_id']]}")

Confirmed excluded from feature_vector.columns: ['health_score', 'priority_score', 'action_type', 'refresh_tier']

Feature vector final columns: ['impressions', 'clicks', 'avg_position', 'sessions', 'engaged_sessions', 'has_ga4', 'ctr', 'pos_1-3', 'pos_11-20', 'pos_21-50', 'pos_4-10', 'pos_51+', 'imp_high', 'imp_low', 'imp_mid']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.